# 🚗 Car Price Prediction Using Machine Learning

## OIBSIP – Data Science Internship

**Task:** Task 3 – Car Price Prediction  
**Objective:** Predict the selling price of a used car using machine learning techniques.

### Project Overview

This project builds machine learning regression models to predict the selling price of used cars based on factors such as car brand, manufacturing year, present price, kilometres driven, fuel type, seller type, transmission, and previous ownership.

The project includes data cleaning, feature engineering, exploratory data analysis, categorical encoding, model training, evaluation, model comparison, and feature importance analysis.

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

# Display plots inside the notebook
%matplotlib inline

print("Libraries imported successfully!")

## 1. Load the Dataset

The CarDekho dataset contains information about used cars and their selling prices. We will first load the dataset and inspect its structure before performing data cleaning and feature engineering.

In [ ]:
# Load the dataset

df = pd.read_csv("dataset/car_data.csv")

# Display the first five rows
df.head()

In [ ]:
# Check the number of rows and columns

print("Dataset Shape:", df.shape)

In [ ]:
# Check column names and data types

df.info()

## 2. Data Understanding and Initial Inspection

Before cleaning the dataset, we inspect its dimensions, data types, missing values, duplicate records, and descriptive statistics. This helps us understand the quality and structure of the data.

In [ ]:
# Check dataset dimensions

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Display column names

print("Column names:")
print(df.columns.tolist())

In [ ]:
# Check data types

print("Data types:")
print(df.dtypes)

In [ ]:
# Check missing values

print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Remove duplicate rows

print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))

In [ ]:
# Check unique values in categorical columns

print("Fuel Types:")
print(df["Fuel_Type"].unique())

print("\nSeller Types:")
print(df["Seller_Type"].unique())

print("\nTransmission Types:")
print(df["Transmission"].unique())

In [ ]:
# Standardize categorical text values

df["Fuel_Type"] = df["Fuel_Type"].str.strip().str.title()
df["Seller_Type"] = df["Seller_Type"].str.strip().str.title()
df["Transmission"] = df["Transmission"].str.strip().str.title()

print("Categorical values standardized successfully.")

## 4. Feature Engineering

Two new features are created to improve the model and satisfy the project requirements:

- **Car_Age:** Calculated from the manufacturing year. Older cars generally have lower resale values.
- **Brand:** Extracted from the `Car_Name` column to capture the effect of the car manufacturer/brand on selling price.

In [ ]:
# Create Car_Age feature

current_year = 2026

df["Car_Age"] = current_year - df["Year"]

# Display the new features

df[["Car_Name", "Year", "Car_Age", "Selling_Price"]].head(10)

In [ ]:
# Improved brand extraction using manufacturer/model keywords

def extract_brand(car_name):
    name = car_name.lower().strip()

    # Maruti Suzuki
    if any(x in name for x in [
        "ritz", "sx4", "ciaz", "wagon r", "swift", "vitara brezza",
        "s cross", "alto", "ertiga", "dzire", "omni", "baleno",
        "ignis", "celerio"
    ]):
        return "Maruti"

    # Toyota
    if any(x in name for x in [
        "innova", "fortuner", "corolla", "etios", "camry",
        "land cruiser"
    ]):
        return "Toyota"

    # Honda
    if any(x in name for x in [
        "honda", "city", "jazz", "amaze", "brio", "wr-v",
        "activa", "cbr", "hornet", "karizma", "cb shine",
        "cb twister"
    ]):
        return "Honda"

    # Hyundai
    if any(x in name for x in [
        "hyundai", "i10", "i20", "verna", "eon", "creta",
        "elantra", "xcent"
    ]):
        return "Hyundai"

    # Mahindra
    if any(x in name for x in [
        "mahindra", "bolero", "scorpio", "xuv", "quanto"
    ]):
        return "Mahindra"

    # Ford
    if any(x in name for x in [
        "ford", "figo", "ecosport", "endeavour", "aspire"
    ]):
        return "Ford"

    # Volkswagen
    if any(x in name for x in [
        "volkswagen", "polo", "vento", "ameo"
    ]):
        return "Volkswagen"

    # Tata
    if any(x in name for x in [
        "tata", "indica", "vista", "manza", "bolt", "tiago",
        "tigor", "nano", "sumo"
    ]):
        return "Tata"

    # Renault
    if any(x in name for x in [
        "renault", "duster", "kwid", "lodgy"
    ]):
        return "Renault"

    # Nissan
    if any(x in name for x in [
        "nissan", "micra", "sunny"
    ]):
        return "Nissan"

    # Chevrolet
    if any(x in name for x in [
        "chevrolet", "beat", "spark", "cruze", "enjoy"
    ]):
        return "Chevrolet"

    # Fiat
    if any(x in name for x in [
        "fiat", "punto", "linea", "avventura"
    ]):
        return "Fiat"

    # KTM
    if "ktm" in name:
        return "KTM"

    # Royal Enfield
    if "royal enfield" in name:
        return "Royal Enfield"

    # Bajaj
    if "bajaj" in name:
        return "Bajaj"

    # Yamaha
    if "yamaha" in name:
        return "Yamaha"

    # TVS
    if "tvs" in name:
        return "TVS"

    # Hero
    if "hero" in name or "splender" in name or "splendor" in name:
        return "Hero"

    # Suzuki motorcycles
    if "suzuki" in name or "access" in name:
        return "Suzuki"
        # Additional models
    if name == "800":
        return "Maruti"

    if "um renegade" in name:
        return "UM"

    if "hyosung" in name:
        return "Hyosung"

    return "Other"



# Apply the improved brand extraction
df["Brand"] = df["Car_Name"].apply(extract_brand)

# Check the results
print("Number of cars by brand:")
print(df["Brand"].value_counts())

In [ ]:
print(df["Brand"].value_counts())


## 5. Exploratory Data Analysis

Exploratory Data Analysis helps us understand the distribution of car prices and relationships between important features before training the machine learning models.

In [ ]:
# Distribution of selling prices

plt.figure(figsize=(10, 6))

sns.histplot(df["Selling_Price"], kde=True)

plt.title("Distribution of Car Selling Prices")
plt.xlabel("Selling Price")
plt.ylabel("Number of Cars")

plt.show()

### Observation

The selling prices are not uniformly distributed. Most cars are concentrated in the lower price range, while a smaller number of cars have substantially higher selling prices. This indicates a right-skewed distribution.

In [ ]:
# Selling price by fuel type

plt.figure(figsize=(8, 6))

sns.boxplot(x="Fuel_Type", y="Selling_Price", data=df)

plt.title("Selling Price by Fuel Type")
plt.xlabel("Fuel Type")
plt.ylabel("Selling Price")

plt.show()

### Observation

The box plot shows how selling prices vary across petrol, diesel, and CNG vehicles. Differences in the median and spread of prices indicate that fuel type can influence the resale value of a vehicle.

In [ ]:
# Relationship between car age and selling price

plt.figure(figsize=(10, 6))

sns.scatterplot(
    x="Car_Age",
    y="Selling_Price",
    data=df
)

plt.title("Car Age vs Selling Price")
plt.xlabel("Car Age (Years)")
plt.ylabel("Selling Price")

plt.show()

### Observation

The scatter plot helps visualize the relationship between vehicle age and selling price. In general, older vehicles tend to have lower selling prices, although other factors such as present price, mileage, fuel type, and brand also influence resale value.

In [ ]:
# Correlation matrix for numerical features

numerical_columns = [
    "Year",
    "Selling_Price",
    "Present_Price",
    "Kms_Driven",
    "Owner",
    "Car_Age"
]

correlation_matrix = df[numerical_columns].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap of Numerical Features")

plt.show()

### Observation

The correlation heatmap shows the strength and direction of relationships between the numerical variables. Selling price is expected to have a strong positive relationship with present price, while car age is generally expected to have a negative relationship with selling price.

## 6. Preparing Data for Machine Learning

The target variable for this project is `Selling_Price`. The remaining relevant features will be used as predictors.

Categorical variables such as fuel type, seller type, transmission, and brand require encoding before they can be used by machine-learning models.

In [ ]:
# Select features and target

features = [
    "Present_Price",
    "Kms_Driven",
    "Fuel_Type",
    "Seller_Type",
    "Transmission",
    "Owner",
    "Car_Age",
    "Brand"
]

X = df[features]
y = df["Selling_Price"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

In [ ]:
# Identify categorical and numerical columns

categorical_features = [
    "Fuel_Type",
    "Seller_Type",
    "Transmission",
    "Brand"
]

numerical_features = [
    "Present_Price",
    "Kms_Driven",
    "Owner",
    "Car_Age"
]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

In [ ]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
# Create preprocessing pipeline for categorical variables

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

print("Preprocessing pipeline created successfully.")

## 7. Model Training

We will train and compare two regression algorithms:

1. **Linear Regression** – a simple baseline regression model.
2. **Random Forest Regressor** – a non-linear ensemble model capable of capturing complex relationships between features.

Both models will be evaluated using MAE, RMSE, and R².

In [ ]:
# Train Linear Regression model

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

# Make predictions
linear_predictions = linear_model.predict(X_test)

print("Linear Regression model trained successfully!")

In [ ]:
# Train Random Forest Regressor

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)

# Make predictions
rf_predictions = random_forest_model.predict(X_test)

print("Random Forest model trained successfully!")

## 8. Model Evaluation

The models will be evaluated using three regression metrics:

- **MAE (Mean Absolute Error):** average absolute difference between actual and predicted prices. Lower is better.
- **RMSE (Root Mean Squared Error):** gives greater weight to larger prediction errors. Lower is better.
- **R² Score:** measures how much variation in selling price is explained by the model. Higher is better.

In [ ]:
# Function to calculate regression metrics

def evaluate_model(model_name, y_true, predictions):
    mae = mean_absolute_error(y_true, predictions)
    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    r2 = r2_score(y_true, predictions)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    }


# Evaluate both models

linear_results = evaluate_model(
    "Linear Regression",
    y_test,
    linear_predictions
)

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    rf_predictions
)

results = pd.DataFrame([
    linear_results,
    rf_results
])

results

## 9. Best Performing Model

Based on the evaluation metrics, **Linear Regression** is the best-performing model overall.

Linear Regression achieved an MAE of **1.5082**, an RMSE of **2.5382**, and an R² score of **0.7500**.

Although Random Forest achieved a slightly lower MAE of **1.4279**, its RMSE was higher at **3.4525** and its R² score was substantially lower at **0.5375**.

Therefore, Linear Regression is selected as the final model because it provides the better overall balance of prediction error and explained variance for this dataset.


## 10. Feature Importance

Linear Regression does not provide a `feature_importances_` attribute like Random Forest. Instead, we use the model's coefficients to understand the influence of each feature.

A larger absolute coefficient indicates a stronger influence on the predicted selling price. Positive coefficients indicate a positive relationship with the prediction, while negative coefficients indicate a negative relationship.

In [ ]:
# Extract feature names after One-Hot Encoding

encoded_feature_names = (
    linear_model.named_steps["preprocessor"]
    .named_transformers_["categorical"]
    .get_feature_names_out(categorical_features)
)

all_feature_names = list(encoded_feature_names) + numerical_features

# Extract Linear Regression coefficients

coefficients = linear_model.named_steps["model"].coef_

feature_importance = pd.DataFrame({
    "Feature": all_feature_names,
    "Coefficient": coefficients
})

feature_importance["Absolute_Coefficient"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

feature_importance.head(15)

### Feature Importance Interpretation

The coefficient analysis shows that fuel type, seller type, transmission, and vehicle characteristics have noticeable influences on the predicted selling price.

Among the encoded variables, `Fuel_Type_Diesel` has the largest positive coefficient (approximately **1.10**). `Fuel_Type_Cng` has a negative coefficient of approximately **-0.79**.

Seller type and transmission also show relatively strong coefficients. Dealer sales have a positive coefficient, while Individual sales have a negative coefficient. Similarly, Automatic transmission has a positive coefficient, while Manual transmission has a negative coefficient.

Among the numerical features, `Present_Price` has a positive coefficient, while `Car_Age` has a negative coefficient. This is consistent with the expectation that older vehicles generally have lower resale values.

The coefficients for categorical variables should be interpreted relative to the reference categories created by One-Hot Encoding. Therefore, they represent relative effects rather than direct changes in selling price.

Overall, the feature analysis shows that fuel type, seller type, transmission, vehicle brand, present price, ownership, and car age contribute to the model's predictions.


In [ ]:
# Plot the top 15 feature influences

top_features = feature_importance.head(15).sort_values(
    "Absolute_Coefficient"
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Coefficient"]
)

plt.title("Top Feature Influences on Car Selling Price")
plt.xlabel("Linear Regression Coefficient")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

## 11. Sample Predictions

The final Linear Regression model can now be used to predict the selling price of used vehicles from their characteristics.

In [ ]:
# Create sample vehicles for prediction

sample_cars = pd.DataFrame({
    "Present_Price": [5.59, 9.54, 6.87],
    "Kms_Driven": [27000, 43000, 42450],
    "Fuel_Type": ["Petrol", "Diesel", "Diesel"],
    "Seller_Type": ["Dealer", "Dealer", "Dealer"],
    "Transmission": ["Manual", "Manual", "Manual"],
    "Owner": [0, 0, 0],
    "Car_Age": [2026 - 2014, 2026 - 2013, 2026 - 2014],
    "Brand": ["Maruti", "Maruti", "Maruti"]
})

# Predict selling prices
sample_predictions = linear_model.predict(sample_cars)

sample_cars["Predicted_Selling_Price"] = sample_predictions

sample_cars

### Prediction Results

The final Linear Regression model successfully generates predicted selling prices for previously unseen vehicle records. The predictions demonstrate how vehicle characteristics such as present price, mileage, fuel type, transmission, ownership, vehicle age, and brand can be used to estimate resale value.

# 12. Conclusion

In this project, a machine learning system was developed to predict the selling price of used vehicles.

The dataset was first cleaned by checking missing values, identifying duplicate records, and standardizing categorical information. Feature engineering was then performed by calculating vehicle age and extracting the vehicle manufacturer from the car model names.

Exploratory Data Analysis was used to understand selling-price distributions and relationships between vehicle characteristics and price. Categorical variables were converted into numerical representations using One-Hot Encoding.

Two regression algorithms were trained and evaluated:

- Linear Regression
- Random Forest Regressor

The models were evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² score.

The final results were:

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Linear Regression | 1.5082 | 2.5382 | 0.7500 |
| Random Forest | 1.4279 | 3.4525 | 0.5375 |

Although Random Forest achieved a slightly lower MAE, Linear Regression achieved a substantially lower RMSE and a much higher R² score. Therefore, **Linear Regression was selected as the best overall model** for this dataset.

The final model achieved an R² score of approximately **0.75**, indicating that it explains around 75% of the variation in selling prices on the test data.

Feature coefficient analysis showed that variables such as fuel type, seller type, transmission, brand, present price, ownership, and car age contributed to the model's predictions.

This project demonstrates a complete machine learning workflow, from data cleaning and exploratory analysis through feature engineering, model training, evaluation, interpretation, and prediction.